In [ ]:
# parameters
# BINDER_FAST: set xi_pts=50 for fast cloud execution
beta = 0.1                                     # junction asymmetry ratio
M = 3                                          # number of large junctions
phi_e_kerr_free = 0.41 * 2 * 3.141592653589793 # operating flux (near Kerr-free point)
E_J = 10.0                                     # large-junction Josephson energy (GHz)
E_C = 0.5                                      # charging energy (GHz)
ga_over_da = 0.05    # hybridisation g_a / Delta_a for cavity Alice (dimensionless)
gb_over_db = 0.05    # hybridisation g_b / Delta_b for cavity Bob   (dimensionless)
xi_max = 3.0         # maximum dimensionless pump amplitude to plot
xi_pts = 100         # number of xi points

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.snail import (
    snail_circuit_params,
    gbs_linear,
    gbs_rwa,
    plot_snail_gbs_vs_xi,
)

## Module 2c: Driven SNAIL — Beam-Splitter Interaction

**Learning objectives:**
- Understand three-wave mixing and the rotating-wave approximation (RWA) that generates a beam-splitter Hamiltonian
- Compute the beam-splitter rate $g_{\rm BS}$ as a function of pump amplitude $|\xi|$
- Understand why the rate deviates from linear at large pump amplitudes
- Identify $\xi_{\rm crit}$ as the practical upper bound on pump strength

---

**Sections:** [1 Three-Wave Mixing](#sec1) · [2 Linear Rate](#sec2) · [3 Full RWA Rate](#sec3) · [4 RWA Breakdown](#sec4)

<a id="sec1"></a>
## 1  Three-Wave Mixing from the SNAIL

### The Hamiltonian

The SNAIL is capacitively coupled to two microwave cavities (Alice, mode $\hat{a}$, frequency $\omega_a$) and (Bob, mode $\hat{b}$, frequency $\omega_b$). The cubic part of the SNAIL potential produces a three-wave mixing (3WM) interaction:

$$\hat{H}_{\rm 3WM} = g_3\,(\hat{\varphi}_a + \hat{\varphi}_b + \hat{\varphi}_p)^3$$

where $\hat{\varphi}_x = \phi_x^{\rm ZPF}(\hat{x} + \hat{x}^\dagger)$ is the zero-point phase of each mode.

### Choosing the pump frequency

By driving the SNAIL with a classical pump at the *difference* frequency $\omega_p = \omega_a - \omega_b$, we select the interaction term $\hat{a}^\dagger\hat{b}$ out of all the cross-terms in the cubic. Replacing the pump mode by its classical coherent amplitude $\hat{\varphi}_p \to \xi e^{-i\omega_p t} + \text{h.c.}$ (where $|\xi|$ is the dimensionless pump amplitude, $\xi = \epsilon_p/\omega_p$) and applying the RWA, we obtain the **beam-splitter Hamiltonian**:

$$\boxed{\hat{H}_{\rm BS} = g_{\rm BS}\left(\hat{a}^\dagger \hat{b} + \hat{a}\hat{b}^\dagger\right)}$$

This conserves total photon number and generates quantum state transfer between Alice and Bob at rate $g_{\rm BS}$.

### Perturbative beam-splitter rate

To leading order in the hybridisation $g_{a,b}/\Delta_{a,b}$ (where $\Delta_x = \omega_x - \omega_c$ is the cavity–coupler detuning), the beam-splitter rate is (Chapman *et al.* Eq. F9):

$$g_{\rm BS}^{(1)} = 6\,\frac{g_a}{\Delta_a}\,\frac{g_b}{\Delta_b}\,g_3\,|\xi|$$

The factor of 6 counts the ways to pick one pump photon and one photon from each cavity from the $\hat{\varphi}^3$ expansion. This rate grows *linearly* with pump amplitude $|\xi|$.

*Lab note: The dimensionless pump amplitude $|\xi| = \epsilon_p/\omega_p$ is controlled by the microwave power delivered to the SNAIL. Typical laboratory values are $|\xi| \approx 0.1$–$1.5$. The coupling prefactors $g_a/\Delta_a$ and $g_b/\Delta_b$ (each $\approx 0.01$–$0.1$) set an overall scale: for $g_3/2\pi \approx 100\,\text{MHz}$ and both ratios $= 0.05$, the beam-splitter rate at $|\xi| = 1$ is $\approx 1.5\,\text{MHz}$.*

In [ ]:
# Get circuit parameters at the Kerr-free flux point
params = snail_circuit_params(beta, M, phi_e_kerr_free, E_J, E_C)

print("SNAIL at Kerr-free operating point:")
print(f"  phi_e    = {phi_e_kerr_free/(2*np.pi):.4f} * 2pi")
print(f"  omega_c  = {params['omega_c']:.4f} GHz")
print(f"  phi_c    = {params['phi_c']:.4f}")
print(f"  g3 / 2pi = {params['g3']*1e3:.2f} MHz")
print(f"  g4 / 2pi = {params['g4']*1e3:.3f} MHz  (should be near zero)")
print(f"  g5 / 2pi = {params['g5']*1e6:.2f} kHz")
print(f"  alpha_c  = {params['alpha_c']*1e3:.3f} MHz")
print(f"  xi_crit  = {params['xi_crit']:.3f}")

<a id="sec2"></a>
## 2  Linear Beam-Splitter Rate

We first compute and plot the perturbative (linear) rate $g_{\rm BS}^{(1)} \propto |\xi|$.

In [ ]:
# Pump amplitude array
xi_arr = np.linspace(0, xi_max, xi_pts)

# Linear (perturbative) beam-splitter rate
gbs_lin = gbs_linear(xi_arr, params['g3'], ga_over_da, gb_over_db)

print(f"Perturbative g_BS at xi = 1.0:  {np.interp(1.0, xi_arr, gbs_lin)*1e3:.2f} MHz")
print(f"Slope d(g_BS)/d|xi| = {6 * ga_over_da * gb_over_db * params['g3'] * 1e3:.4f} MHz / unit xi")

# Plot
fig_lin, ax_lin = plt.subplots(figsize=(6, 4))
ax_lin.plot(xi_arr, gbs_lin * 1e3, lw=1.5, color='C0', label=r"$g_{\rm BS}^{(1)}$ (linear)")
ax_lin.set_xlabel(r"Pump amplitude $|\xi|$")
ax_lin.set_ylabel(r"$g_{\rm BS}/2\pi$ (MHz)")
ax_lin.set_title("Perturbative beam-splitter rate")
ax_lin.legend(fontsize=9, frameon=False)
ax_lin.tick_params(direction='in', top=True, right=True)

# Annotate the slope
slope_mhz = 6 * ga_over_da * gb_over_db * params['g3'] * 1e3
ax_lin.annotate(
    rf"slope $= 6\,(g_a/\Delta_a)(g_b/\Delta_b)\,g_3 = {slope_mhz:.3f}\,\mathrm{{MHz}}$",
    xy=(1.0, np.interp(1.0, xi_arr, gbs_lin * 1e3)),
    xytext=(0.6, np.interp(1.0, xi_arr, gbs_lin * 1e3) * 0.6),
    fontsize=8,
    arrowprops=dict(arrowstyle='->', color='gray'),
)
plt.tight_layout()
plt.show()

*Lab note: The linear scaling $g_{\rm BS} \propto |\xi|$ is the first design principle: doubling the pump power (field amplitude $\propto \sqrt{P}$, so $|\xi| \propto \sqrt{P}$) increases the rate by $\sqrt{2}$. In practice, the experimentalist adjusts the pump power to reach the target beam-splitter rate without exceeding the critical pump amplitude — see Section 4.*

<a id="sec3"></a>
## 3  Full RWA Rate: Higher-Order Corrections

The perturbative formula breaks down when higher-order terms in the Taylor expansion (especially $g_5$) become important. The full RWA result sums over *all* odd nonlinearities:

$$g_{\rm BS}^{\rm RWA} = \frac{g_a}{\Delta_a}\,\frac{g_b}{\Delta_b}\,|\xi|\sum_{m=1}^{\infty} C_m\, g_{2m+1}\,|\xi|^{2(m-1)}$$

where $C_m = \dfrac{(2m+1)!}{m!\,(m-1)!}$ counts the number of RWA-resonant contractions from the $(2m+1)$-th order term (Chapman *et al.* Eq. F10). The $m=1$ term recovers the linear result; $m=2$ introduces $g_5|\xi|^2$ corrections, and so on.

**Rate turnaround.** Because $g_5 < 0$ at typical operating points (it is proportional to $c_5 < 0$ for $M=3$, $\phi_e < \pi$), the $m=2$ term *subtracts* from the $m=1$ term at large $|\xi|$. The rate reaches a maximum near:

$$|\xi|_{\rm crit} = \frac{3\pi}{2\phi_c}$$

Beyond this point the pump amplitude causes the SNAIL to sample regions far from the minimum and the rate decreases — the physical interpretation is that the Bessel-function envelope of the coherent drive starts to suppresses the cubic matrix element.

*Lab note: Typical values: $\xi_{\rm crit} \approx 1$–$3$ depending on $\phi_c$. The beam-splitter rate at $\xi_{\rm crit}$ is approximately $g_{\rm BS,max} \approx 10$–$50\,\text{MHz}$, consistent with observed quantum state transfer rates in SNAIL experiments.*

In [ ]:
# Full RWA beam-splitter rate (include g3 and g5)
gbs_rwa_arr = gbs_rwa(
    xi_arr,
    g_odd=[params['g3'], params['g5']],
    ga_over_delta_a=ga_over_da,
    gb_over_delta_b=gb_over_db,
)

xi_crit = params['xi_crit']
print(f"xi_crit = {xi_crit:.3f}")
print(f"g_BS(xi_crit) = {np.interp(xi_crit, xi_arr, gbs_rwa_arr)*1e3:.2f} MHz  (RWA)")
print(f"g_BS(xi_crit) = {np.interp(xi_crit, xi_arr, gbs_lin)*1e3:.2f} MHz  (linear)")

In [ ]:
# Compare linear and RWA rates on the same plot
fig_rwa, ax_rwa = plot_snail_gbs_vs_xi(
    xi_arr,
    [gbs_lin, gbs_rwa_arr],
    labels=[r"linear $g_{\rm BS}^{(1)}$", r"RWA $g_{\rm BS}$"],
    colors=['C0', 'C1'],
    units='MHz',
)

# Mark the critical pump amplitude
ax_rwa.axvline(xi_crit, color='crimson', ls='--', lw=1.0,
               label=rf"$|\xi_{{\rm crit}}| = {xi_crit:.2f}$")
ax_rwa.legend(fontsize=8, frameon=False)
ax_rwa.set_title(
    rf"Beam-splitter rate, $\beta={beta}$, $M={M}$, "
    rf"$g_a/\Delta_a = g_b/\Delta_b = {ga_over_da}$"
)
plt.show()

In [ ]:
# Relative deviation of linear from RWA (measure of g5 correction)
fig_dev, ax_dev = plt.subplots(figsize=(6, 4))

# Avoid division by zero at xi=0
mask = gbs_rwa_arr > 0
deviation = (gbs_lin[mask] - gbs_rwa_arr[mask]) / gbs_rwa_arr[mask] * 100

ax_dev.plot(xi_arr[mask], deviation, lw=1.5, color='C3')
ax_dev.axhline(0, color='k', lw=0.5, ls='--')
ax_dev.axhline(10, color='gray', lw=0.5, ls=':', label='10% deviation')
ax_dev.axvline(xi_crit, color='crimson', ls='--', lw=1.0,
               label=rf"$\xi_{{\rm crit}} = {xi_crit:.2f}$")
ax_dev.set_xlabel(r"Pump amplitude $|\xi|$")
ax_dev.set_ylabel(r"$(g_{\rm BS}^{(1)} - g_{\rm BS}^{\rm RWA}) / g_{\rm BS}^{\rm RWA}$ (%)")
ax_dev.set_title("Perturbative vs RWA deviation")
ax_dev.legend(fontsize=8, frameon=False)
ax_dev.tick_params(direction='in', top=True, right=True)
plt.tight_layout()
plt.show()

# Find xi where deviation exceeds 10%
idx_10 = np.where(deviation > 10)[0]
if len(idx_10) > 0:
    print(f"Linear formula deviates >10% above xi ~ {xi_arr[mask][idx_10[0]]:.3f}")
else:
    print("Linear formula remains within 10% over the plotted range.")

<a id="sec4"></a>
## 4  RWA Breakdown and Practical Constraints

### Why the rate turns over

At pump amplitudes $|\xi| \gg \xi_{\rm crit}$, the coherent displacement of the SNAIL potential is so large that the Taylor expansion in $\varphi$ is no longer valid. Physically, the pump drives the phase variable far from the potential minimum, and the Josephson cosine becomes highly nonlinear. The full result can be expressed as a Bessel-function expansion whose envelope peaks at $|\xi| \approx \xi_{\rm crit}$ and decreases for larger pump.

Practically, exceeding $\xi_{\rm crit}$ also risks:
1. **Loss of single-well confinement:** at very large phase excursions the potential barrier can be overcome
2. **Strong parametric instability:** off-resonant processes activate as $|\xi|$ grows
3. **Heating and quasiparticle generation** from large voltage swings across the junctions

### Practical upper bound on $g_{\rm BS}$

The maximum achievable beam-splitter rate is approximately:

$$g_{\rm BS}^{\rm max} \approx g_{\rm BS}^{\rm RWA}(\xi_{\rm crit})$$

For the parameters in this notebook:

$$g_{\rm BS}^{\rm max}/2\pi \approx \text{(see printed value above)}$$

Typical measured values in published SNAIL beam-splitter experiments are $g_{\rm BS}/2\pi \approx 0.5$–$20\,\text{MHz}$, consistent with these estimates.

### Flux dependence of the maximum rate

Since both $g_3$ and $\xi_{\rm crit}$ depend on flux, $g_{\rm BS}^{\rm max}$ is also flux-tunable. Near the Kerr-free point the product $g_3 \times \xi_{\rm crit} = g_3 \times 3\pi/(2\phi_c)$ is nearly maximised because $g_3$ peaks there while $\phi_c$ is still moderate.

*Lab note: In experiments, the pump amplitude is typically calibrated by measuring the vacuum Rabi splitting between Alice and Bob as a function of pump power. The linear regime $g_{\rm BS} \propto \sqrt{P_{\rm pump}}$ is used for calibration, and $\xi_{\rm crit}$ is identified as the pump power where the rate stops growing as expected.*

In [ ]:
# Maximum beam-splitter rate summary
gbs_max = np.max(gbs_rwa_arr)
xi_at_max = xi_arr[np.argmax(gbs_rwa_arr)]

print("Summary of beam-splitter rates at the Kerr-free operating point:")
print(f"  g3 / 2pi         = {params['g3']*1e3:.2f} MHz")
print(f"  xi_crit          = {xi_crit:.3f}")
print(f"  xi at RWA peak   = {xi_at_max:.3f}")
print(f"  g_BS^(1)(xi_crit)  = {np.interp(xi_crit, xi_arr, gbs_lin)*1e3:.2f} MHz (linear)")
print(f"  g_BS^RWA(xi_crit)  = {np.interp(xi_crit, xi_arr, gbs_rwa_arr)*1e3:.2f} MHz (RWA)")
print(f"  g_BS^RWA maximum   = {gbs_max*1e3:.2f} MHz  at xi = {xi_at_max:.3f}")
print()
print(f"Coupling prefactor g_a/Da * g_b/Db = {ga_over_da * gb_over_db:.4f}")